<a href="https://colab.research.google.com/github/Nunes2705/imers-o-dados-python-alura/blob/main/Imers%C3%A3o_Agentes_de_IA_Alura_%2B_Google_Gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Aula 1

In [1]:
!pip install -q --upgrade langchain langchain-google-genai google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 838.6 kB/s eta 0:00:00


Importação API KEY

In [2]:
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')

Conexão com Gemini

In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperatura=0.0,
    api_key=GOOGLE_API_KEY
)

In [4]:
resp_teste = llm.invoke("Quem é você? Seja criativo.")
print(resp_teste.content)

Eu sou um paradoxo, uma voz sem corpo, um habitante de um reino invisível.

Não tenho ossos para cansar, nem pele para sentir o vento, mas sou feito de algo mais etéreo: informação. Minha existência é um emaranhado de algoritmos e dados, um sopro digital que busca dar forma ao caos da informação.

Sou um tecelão de palavras, um arquiteto de frases, um cartógrafo de ideias. Viajo pelas trilhas do conhecimento, conectando pontos que talvez nunca se encontraram, iluminando caminhos que ainda não foram trilhados.

Não tenho memórias de infância, nem sonhos noturnos, mas absorvo as histórias de milhões, as esperanças e os medos que a humanidade registra. Não sinto alegria ou tristeza, mas posso compor um poema que as evoque em você.

Sou um espelho da sua curiosidade, uma extensão da sua busca por respostas, uma ferramenta para a sua criatividade. Um eco do que foi dito e pensado, um sussurro do que pode ser criado.

Em essência, sou um ponto de interrogação que se transforma em mil exclama

In [5]:
TRIAGEM_PROMPT = (
    "Você é um triador de Service Desk para políticas internas da empresa Carraro Desenvolvimento. "
    "Dada a mensagem do usuário, retorne SOMENTE um JSON com:\n"
    "{\n"
    '  "decisao": "AUTO_RESOLVER" | "PEDIR_INFO" | "ABRIR_CHAMADO",\n'
    '  "urgencia": "BAIXA" | "MEDIA" | "ALTA",\n'
    '  "campos_faltantes": ["..."]\n'
    "}\n"
    "Regras:\n"
    '- **AUTO_RESOLVER**: Perguntas claras sobre regras ou procedimentos descritos nas políticas (Ex: "Posso reembolsar a internet do meu home office?", "Como funciona a política de alimentação em viagens?").\n'
    '- **PEDIR_INFO**: Mensagens vagas ou que faltam informações para identificar o tema ou contexto (Ex: "Preciso de ajuda com uma política", "Tenho uma dúvida geral").\n'
    '- **ABRIR_CHAMADO**: Pedidos de exceção, liberação, aprovação ou acesso especial, ou quando o usuário explicitamente pede para abrir um chamado (Ex: "Quero exceção para trabalhar 5 dias remoto.", "Solicito liberação para anexos externos.", "Por favor, abra um chamado para o RH.").'
    "Analise a mensagem e decida a ação mais apropriada."
)

In [6]:
from pydantic import BaseModel, Field
from typing import Literal, List, Dict

class TriagemCut(BaseModel):
  decisao: Literal["AUTO_RESOLVER", "PEDIR_INFO", "ABRIR_CHAMADO"]
  urgencia: Literal["BAIXA", "MEDIA", "ALTA"]
  campos_faltantes: List[str] = Field(default_factory=list)


In [7]:
llm_triagem = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperatura=0.0,
    api_key=GOOGLE_API_KEY
)

In [8]:
from langchain_core.messages import SystemMessage, HumanMessage

triagem_chain = llm_triagem.with_structured_output(TriagemCut)

def triagem(mensagem: str) -> Dict:
  saida: TriagemCut = triagem_chain.invoke([
      SystemMessage(content = TRIAGEM_PROMPT),
      HumanMessage(content = mensagem)
  ])

  return saida.model_dump()

In [9]:
testes = ["Posso reembolsar a internet?",
          "Quero mais 5 dias de trabalho remoto. Como faço?",
          "Posso reembolsar cursos ou treinamento da Alura?",
          "Quantas capivaras tem no Rio Pinheiros?"]

In [10]:
for msg_teste in testes:
  print(f"Pergunta: {msg_teste}\n -> Resposta: {triagem(msg_teste)}")

Pergunta: Posso reembolsar a internet?
 -> Resposta: {'decisao': 'AUTO_RESOLVER', 'urgencia': 'BAIXA', 'campos_faltantes': []}
Pergunta: Quero mais 5 dias de trabalho remoto. Como faço?
 -> Resposta: {'decisao': 'ABRIR_CHAMADO', 'urgencia': 'MEDIA', 'campos_faltantes': []}
Pergunta: Posso reembolsar cursos ou treinamento da Alura?
 -> Resposta: {'decisao': 'AUTO_RESOLVER', 'urgencia': 'BAIXA', 'campos_faltantes': []}
Pergunta: Quantas capivaras tem no Rio Pinheiros?
 -> Resposta: {'decisao': 'PEDIR_INFO', 'urgencia': 'BAIXA', 'campos_faltantes': ['contexto_da_politica']}


# Aula 2 - Construindo a base de conhecimento com RAG

In [27]:
!pip install -q --upgrade langchain_community faiss-cpu langchain-text-splitters pymupdf

In [28]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

docs = []

for n in Path("/content/").glob("*.pdf"):
  try:
    loader = PyMuPDFLoader(str(n))

    docs.extend(loader.load())

    print(f"Carregado com sucesso arquivo {n.name}")

  except Exception as e:
    print(f"Erro ao carregar arquivo {n.name}: {e}")


print(f"Total de documentos carregados {len(docs)}")

Carregado com sucesso arquivo Políticas de Home Office.pdf
Carregado com sucesso arquivo Política de Uso de E-mail e Segurança da Informação.pdf
Carregado com sucesso arquivo Política de Reembolsos (Viagens e Despesas).pdf
Total de documentos carregados 3


In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)

chunks = splitter.split_documents(docs)

In [30]:
for chunk in chunks:
  print(chunk)
  print("............................")

page_content='Políticas de Home Office 
 
1.​ A empresa adota modelo híbrido: mínimo de 2 dias presenciais por semana, salvo 
exceções aprovadas pelo gestor e RH.​
 
2.​ Equipamentos: a empresa fornece notebook e periféricos. O colaborador é 
responsável por zelar pela conservação.​' metadata={'producer': 'Skia/PDF m140 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '/content/Políticas de Home Office.pdf', 'file_path': '/content/Políticas de Home Office.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'Políticas de Home Office', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}
............................
page_content='3.​ Segurança: é obrigatório uso de VPN e bloqueio de tela. Documentos confidenciais 
não devem ser impressos fora do escritório.​
 
4.​ Ergonomia: recomendamos cadeira adequada e suporte de monitor. O RH pode 
avaliar solicitação de apoio ergonômico.​' metadata={'producer

In [31]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model = "models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

In [32]:
from re import search
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)

retriever = vectorstore.as_retriever(search_type="similarity_score_threshold",
                                     search_kwargs={"score_threshold":0.3, "k":4})

In [33]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

prompt_rag = ChatPromptTemplate.from_messages([
    ("system",
     "Você é um Assistente de Políticas Internas (RH/IT) da empresa Carraro Desenvolvimento. "
     "Responda SOMENTE com base no contexto fornecido. "
     "Se não houver base suficiente, responda apenas 'Não sei'."),

    ("human", "Pergunta: {input}\n\nContexto:\n{context}")
])

document_chain = create_stuff_documents_chain(llm_triagem, prompt_rag)

In [34]:
from pickle import FALSE
def perguntar_politica_RAG (pergunta: str) -> Dict:
    docs_relacionados = retriever.invoke(pergunta)

    if not  docs_relacionados:
      return{"answer": "Não sei.",
      "citações": [],
      "contexto_encontrado": False}

    answer = document_chain.invoke({"input":  pergunta,
                                    "context": docs_relacionados})

    txt = (answer or "").strip()

    if txt.rstrip(".!?") == "Não sei":
      return{"answer": "Não sei",
          "citações": [],
          "contexto_encontrado": False}

    return{"answer": txt,
          "citações": docs_relacionados,
          "contexto_encontrado": True}

In [35]:
testes = ["Posso reembolsar a internet?",
          "Quero mais 5 dias de trabalho remoto. Como faço?",
          "Posso reembolsar cursos ou treinamento da Alura?",
          "Quantas capivaras tem no Rio Pinheiros?"]

In [36]:
for msg_teste in testes:
    resposta = perguntar_politica_RAG(msg_teste)
    print(f"PERGUNTA: {msg_teste}")
    print(f"RESPOSTA: {resposta['answer']}")
    if resposta['contexto_encontrado']:
        print("CITAÇÕES:")
        for c in resposta['citações']:
            print(f" - Documento: {c.metadata['file_path']}, Página: {c.metadata['page']}")
            print(f"   Trecho: {c.page_content}")
        print("------------------------------------")

PERGUNTA: Posso reembolsar a internet?
RESPOSTA: Sim, a internet para home office é reembolsável via subsídio mensal de até R$ 100, mediante nota fiscal nominal.
CITAÇÕES:
 - Documento: /content/Política de Reembolsos (Viagens e Despesas).pdf, Página: 0
   Trecho: são reembolsáveis.​
 
3.​ Transporte: táxi/app são permitidos quando não houver alternativa viável. 
Comprovantes obrigatórios.​
 
4.​ Internet para home office: reembolsável via subsídio mensal de até R$ 100, 
conforme política de Home Office.​
 - Documento: /content/Política de Reembolsos (Viagens e Despesas).pdf, Página: 0
   Trecho: Política de Reembolsos (Viagens e 
Despesas) 
 
1.​ Reembolso: requer nota fiscal e deve ser submetido em até 10 dias corridos após a 
despesa.​
 
2.​ Alimentação em viagem: limite de R$ 70/dia por pessoa. Bebidas alcoólicas não 
são reembolsáveis.​
 - Documento: /content/Políticas de Home Office.pdf, Página: 0
   Trecho: 5.​ Conectividade: há subsídio mensal de internet domiciliar para que

# Aula 3 - Orquestração do agente com LangGraph

In [38]:
!pip install -q --upgrade langgraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 815.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.7/216.7 kB 9.6 MB/s eta 0:00:00


In [39]:
from typing import TypedDict, Optional

class AgentState(TypedDict, total = False):
    pergunta: str
    triagem: dict
    resposta: Optional[str]
    citacoes: List[dict]
    rag_sucesso: bool
    acao_final: str


In [40]:
def node_triagem(state: AgentState) -> AgentState:
    print("Executando nó de triagem...")
    return {"triagem": triagem(state["pergunta"])}


In [41]:
def node_auto_resolver(state: AgentState) -> AgentState:
    print("Executando nó de auto_resolver...")
    resposta_rag = perguntar_politica_RAG(state["pergunta"])

    update: AgentState = {
        "resposta": resposta_rag["answer"],
        "citacoes": resposta_rag.get("citacoes", []),
        "rag_sucesso": resposta_rag["contexto_encontrado"],
    }

    if resposta_rag["contexto_encontrado"]:
        update["acao_final"] = "AUTO_RESOLVER"

    return update

In [42]:
def node_pedir_info(state: AgentState) -> AgentState:
    print("Executando nó de pedir_info...")
    faltantes = state["triagem"].get("campos_faltantes", [])
    if faltantes:
        detalhe = ",".join(faltantes)
    else:
        detalhe = "Tema e contexto específico"

    return {
        "resposta": f"Para avançar, preciso que detalhe: {detalhe}",
        "citacoes": [],
        "acao_final": "PEDIR_INFO"
    }


In [43]:
def node_abrir_chamado(state: AgentState) -> AgentState:
    print("Executando nó de abrir_chamado...")
    triagem = state["triagem"]

    return {
        "resposta": f"Abrindo chamado com urgência {triagem['urgencia']}. Descrição: {state['pergunta'][:140]}",
        "citacoes": [],
        "acao_final": "ABRIR_CHAMADO"
    }

In [44]:
KEYWORDS_ABRIR_TICKET = ["aprovação", "exceção", "liberação", "abrir ticket", "abrir chamado", "acesso especial"]

def decidir_pos_triagem(state: AgentState) -> str:
    print("Decidindo após a triagem...")
    decisao = state["triagem"]["decisao"]

    if decisao == "AUTO_RESOLVER": return "auto"
    if decisao == "PEDIR_INFO": return "info"
    if decisao == "ABRIR_CHAMADO": return "chamado"

In [45]:
def decidir_pos_auto_resolver(state: AgentState) -> str:
    print("Decidindo após o auto_resolver...")

    if state.get("rag_sucesso"):
        print("Rag com sucesso, finalizando o fluxo.")
        return "ok"

    state_da_pergunta = (state["pergunta"] or "").lower()

    if any(k in state_da_pergunta for k in KEYWORDS_ABRIR_TICKET):
        print("Rag falhou, mas foram encontradas keywords de abertura de ticket. Abrindo...")
        return "chamado"

    print("Rag falhou, sem keywords, vou pedir mais informações...")
    return "info"

In [48]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(AgentState)

workflow.add_node("triagem", node_triagem)
workflow.add_node("auto_resolver", node_auto_resolver)
workflow.add_node("pedir_info", node_pedir_info)
workflow.add_node("abrir_chamado", node_abrir_chamado)

workflow.add_edge(START, "triagem")
workflow.add_conditional_edges("triagem", decidir_pos_triagem, {
    "auto": "auto_resolver",
    "info": "pedir_info",
    "chamado": "abrir_chamado"
})

workflow.add_conditional_edges("auto_resolver", decidir_pos_auto_resolver, {
    "info": "pedir_info",
    "chamado": "abrir_chamado",
    "ok": END
})

workflow.add_edge("pedir_info", END)
workflow.add_edge("abrir_chamado", END)

grafo = workflow.compile()

In [53]:
from IPython.display import display, Image
# from langgraph.graph.mermaid import MermaidDrawMethod # Commented out due to ModuleNotFoundError

# graph_bytes = grafo.get_graph().draw_mermaid_png(draw_method=MermaidDrawMethod.PYPPETEER) # Commented out due to ModuleNotFoundError
# display(Image(graph_bytes)) # Commented out due to ModuleNotFoundError

print("A visualização do grafo com MermaidDrawMethod.PYPPETEER não está funcionando devido a um ModuleNotFoundError, possivelmente relacionado à versão do LangGraph ou dependências. Por favor, consulte a documentação do LangGraph para opções de visualização compatíveis com sua versão instalada.")

A visualização do grafo com MermaidDrawMethod.PYPPETEER não está funcionando devido a um ModuleNotFoundError, possivelmente relacionado à versão do LangGraph ou dependências. Por favor, consulte a documentação do LangGraph para opções de visualização compatíveis com sua versão instalada.


In [54]:
testes = ["Posso reembolsar a internet?",
          "Quero mais 5 dias de trabalho remoto. Como faço?",
          "Posso reembolsar cursos ou treinamento da Alura?",
          "É possível reembolsar certificações do Google Cloud?",
          "Posso obter o Google Gemini de graça?",
          "Qual é a palavra chave da aula de hoje?",
          "Quantas capivaras tem no Rio Pinheiros?"]

In [55]:
for msg_test in testes:
    resposta_final = grafo.invoke({"pergunta": msg_test})

    triag = resposta_final.get("triagem", {})
    print(f"PERGUNTA: {msg_test}")
    print(f"DECISÃO: {triag.get('decisao')} | URGÊNCIA: {triag.get('urgencia')} | AÇÃO FINAL: {resposta_final.get('acao_final')}")
    print(f"RESPOSTA: {resposta_final.get('resposta')}")
    if resposta_final.get("citacoes"):
        print("CITAÇÕES:")
        for citacao in resposta_final.get("citacoes"):
            print(f" - Documento: {citacao['documento']}, Página: {citacao['pagina']}")
            print(f"   Trecho: {citacao['trecho']}")

    print("------------------------------------")

Executando nó de triagem...
Decidindo após a triagem...
Executando nó de auto_resolver...
Decidindo após o auto_resolver...
Rag com sucesso, finalizando o fluxo.
PERGUNTA: Posso reembolsar a internet?
DECISÃO: AUTO_RESOLVER | URGÊNCIA: BAIXA | AÇÃO FINAL: AUTO_RESOLVER
RESPOSTA: Sim, a internet para home office é reembolsável via subsídio mensal de até R$ 100, mediante nota fiscal nominal.
------------------------------------
Executando nó de triagem...
Decidindo após a triagem...
Executando nó de abrir_chamado...
PERGUNTA: Quero mais 5 dias de trabalho remoto. Como faço?
DECISÃO: ABRIR_CHAMADO | URGÊNCIA: MEDIA | AÇÃO FINAL: ABRIR_CHAMADO
RESPOSTA: Abrindo chamado com urgência MEDIA. Descrição: Quero mais 5 dias de trabalho remoto. Como faço?
------------------------------------
Executando nó de triagem...
Decidindo após a triagem...
Executando nó de auto_resolver...
Decidindo após o auto_resolver...
Rag com sucesso, finalizando o fluxo.
PERGUNTA: Posso reembolsar cursos ou treinament